# Unidade 3 · Outras famílias de modelos
**Aprendizagem Computacional · 4º TIA · Apostila do Aluno · versão INICIAL**

Este notebook acompanha a **Unidade 3** da apostila (cap. 11 a 16). O problema é o da Unidade 2 (tumor maligno ou benigno, `load_breast_cancer`), com o **mesmo protocolo**: o mesmo split, as mesmas 5 dobras e as mesmas métricas (F1 e recall da classe maligna). Cada seção acrescenta uma família de modelos e mostra o seu **botão contra o sobreajuste**:

| Seção | Família | Botão | Capítulo |
|---|---|---|---|
| 1 | k-NN (vizinhança) | k | 11 |
| 2 | Naive Bayes (probabilidade) | — | 12 |
| 3 | Árvore de decisão (perguntas) | profundidade | 13 |
| 4 | SVM (a fronteira mais segura) | C e gamma | 14 |
| 5 | Rede neural MLP (transformação aprendida) | tamanho e alpha | 15 |
| 6 | Ensembles e a comparação justa | nº de árvores | 16 |

No fim, escolhemos **um** modelo e abrimos o teste uma única vez.

**Como usar a versão INICIAL:** procure os comentários `# TODO`. Complete o código e rode a célula. As células seguintes dependem das anteriores.

## 0 · O ponto de partida: dados, protocolo e a régua
Mesmo split e mesmas dobras da Unidade 2. A tabela começa com o baseline e a logística, e cada seção acrescenta uma linha.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)  # o baseline nunca prevê 'maligno'
SEMENTE = 42
pd.set_option('display.precision', 3)

from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)
Xc = bc.data
yc = (bc.target == 0).astype(int)   # 1 = maligno
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=SEMENTE)
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

def nota(modelo):
    """O mesmo protocolo para todos: 5 dobras estratificadas no treino, F1 e recall da classe maligna."""
    r = cross_validate(modelo, Xc_tr, yc_tr, cv=cv_c, scoring=['f1', 'recall'])
    return {'F1': round(float(r['test_f1'].mean()), 3), 'desvio F1': round(float(r['test_f1'].std()), 3),
            'recall': round(float(r['test_recall'].mean()), 3)}

def busca(modelo, grade):
    return GridSearchCV(modelo, grade, cv=cv_c, scoring='f1')

modelos = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logística (cap. 9)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
}
print(Xc.shape, '| malignos no treino:', yc_tr.sum(), 'de', len(yc_tr))
pd.DataFrame({n: nota(m) for n, m in modelos.items()}).T

## 1 · k-NN: decidir pela vizinhança (cap. 11)
O k-NN não ajusta coeficientes: guarda o treino e, para cada caso novo, vota com os k vizinhos mais próximos. Precisa de escala (cap. 7). O `GridSearchCV` escolhe o k **dentro** do treino.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# TODO 1 (cap. 11): k-NN com StandardScaler num Pipeline, buscando n_neighbors em [3, 5, 7, 11, 15]
modelos['k-NN (cap. 11)'] = None
knn = modelos['k-NN (cap. 11)'].fit(Xc_tr, yc_tr)
print('k escolhido:', knn.best_params_['m__n_neighbors'])
print(nota(modelos['k-NN (cap. 11)']))

**Leitura:** k pequeno decora (um caso mal rotulado contamina a vizinhança); k grande demais vira uma média de tudo. O k escolhido é o meio-termo que a validação cruzada encontrou.

## 2 · Naive Bayes: decidir por probabilidade (cap. 12)
O Naive Bayes calcula P(maligno | medidas) supondo que as medidas são **independentes** entre si, dada a classe. A suposição é "ingênua" (raio, perímetro e área andam juntos), mas o modelo é rápido e muitas vezes vai bem.

In [ ]:
from sklearn.naive_bayes import GaussianNB
modelos['Naive Bayes (cap. 12)'] = make_pipeline(StandardScaler(), GaussianNB())
print(nota(modelos['Naive Bayes (cap. 12)']))
print('Correlação entre raio, perímetro e área médios:')
Xc_tr[['mean radius', 'mean perimeter', 'mean area']].corr().round(3)

**Leitura:** raio, perímetro e área têm correlação perto de 1, o oposto da independência que o modelo supõe. O Naive Bayes "conta" a mesma evidência três vezes e fica confiante demais. Mesmo assim, a nota é razoável: a suposição errada atrapalha as probabilidades mais do que a classificação.

## 3 · Árvore de decisão: decidir por perguntas (cap. 13)
A árvore faz uma pergunta de sim ou não por vez. Não precisa de escala e dá para ler. O botão é a profundidade: na Unidade 2 vimos que, sem limite, ela decora o treino.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
# TODO 2 (cap. 13): árvore com random_state=SEMENTE, buscando max_depth em [2, 3, 4, 5, 6, None]
modelos['Árvore (cap. 13)'] = None
arvore = modelos['Árvore (cap. 13)'].fit(Xc_tr, yc_tr)
print('profundidade escolhida:', arvore.best_params_['max_depth'])
print(nota(modelos['Árvore (cap. 13)']))
rasa = DecisionTreeClassifier(max_depth=2, random_state=SEMENTE).fit(Xc_tr, yc_tr)
print(export_text(rasa, feature_names=list(Xc.columns)))

**Leitura:** a árvore de 2 níveis cabe numa frase que um médico entende: "se o pior perímetro passa de 112,8 e a textura média passa de 14,95, maligno; abaixo de 112,8, maligno só se os pontos côncavos passam de 0,18". É a família mais explicável, e também a mais instável: mudar poucas linhas do treino pode trocar a primeira pergunta.

## 4 · SVM: a fronteira mais segura (cap. 14)
Entre todas as fronteiras que separam as classes, o SVM escolhe a de **maior margem**. O C controla quanto erro no treino se tolera (C grande = margem estreita, risco de decorar); o kernel RBF permite fronteiras curvas, e o gamma diz quão curvas.

In [ ]:
from sklearn.svm import SVC
modelos['SVM RBF (cap. 14)'] = busca(Pipeline([('esc', StandardScaler()), ('m', SVC(random_state=SEMENTE))]),
                                     {'m__C': [0.1, 1, 10], 'm__gamma': ['scale', 0.01]})
svm = modelos['SVM RBF (cap. 14)'].fit(Xc_tr, yc_tr)
print('melhores:', svm.best_params_)
print('vetores de suporte (os casos que definem a fronteira):', svm.best_estimator_[-1].n_support_.sum(), 'de', len(Xc_tr))
print(nota(modelos['SVM RBF (cap. 14)']))

**Leitura:** só uma parte dos casos do treino define a fronteira: os **vetores de suporte**, que ficam perto dela. Os outros poderiam sair do treino sem mudar nada.

## 5 · Rede neural MLP: a transformação aprendida (cap. 15)
Uma rede é um empilhado de logísticas: a camada escondida aprende sozinha uma transformação das 30 medidas, e a saída decide. O `alpha` é a regularização (como o Ridge); o número de neurônios é a capacidade.

In [ ]:
from sklearn.neural_network import MLPClassifier
modelos['MLP (cap. 15)'] = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-3,
                                                                         max_iter=2000, random_state=SEMENTE))
print(nota(modelos['MLP (cap. 15)']))

**Leitura:** numa tabela com 30 medidas que já têm significado, a rede fica perto da logística, não muito acima. Redes brilham quando o dado é bruto (pixels, som, texto) e é preciso construir as variáveis, como nos notebooks Extra de visão computacional.

## 6 · Ensembles e a comparação justa (cap. 16)
Cada família erra de um jeito. Combinar modelos diferentes cancela parte desses erros:
- **Random Forest:** muitas árvores, cada uma num sorteio do treino, votando;
- **Gradient Boosting:** árvores pequenas em fila, cada uma corrigindo a anterior;
- **Votação suave:** a média das probabilidades de modelos de famílias diferentes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
modelos['Random Forest (cap. 16)'] = RandomForestClassifier(n_estimators=300, random_state=SEMENTE)
modelos['Gradient Boosting (cap. 16)'] = GradientBoostingClassifier(random_state=SEMENTE)
modelos['Votação suave (cap. 16)'] = VotingClassifier(
    [('lr', modelos['Logística (cap. 9)']), ('mlp', modelos['MLP (cap. 15)']),
     ('rf', modelos['Random Forest (cap. 16)'])], voting='soft')

linhas = []
for nome, m in modelos.items():
    r = cross_validate(m, Xc_tr, yc_tr, cv=cv_c, scoring=['f1', 'recall', 'precision', 'roc_auc'])
    linhas.append({'modelo': nome, 'F1': r['test_f1'].mean(), 'desvio F1': r['test_f1'].std(),
                   'recall': r['test_recall'].mean(), 'precisão': r['test_precision'].mean(),
                   'ROC-AUC': r['test_roc_auc'].mean()})
comparacao = pd.DataFrame(linhas).sort_values('F1', ascending=False).reset_index(drop=True)
comparacao

**Como ler a tabela:** quando as diferenças de F1 entre os primeiros colocados são menores que o desvio entre dobras, eles estão **empatados** na prática. Nesse caso, desempate por simplicidade e interpretabilidade, e não pela terceira casa decimal. Note também que o baseline tem F1 zero: ele nunca aponta um maligno.

**Pergunta para discussão:** a árvore isolada fica abaixo da Random Forest. Por quê?

**Resposta esperada:** uma árvore sozinha tem variância alta (muda muito com pequenas mudanças nos dados). A Random Forest faz a média de centenas de árvores treinadas em amostras e colunas sorteadas, o que reduz essa variância (cap. 16).

## 7 · Escolher e abrir o teste UMA vez
Escolhemos a **regressão logística**: ela fica entre as melhores na validação cruzada, é simples, gera probabilidades bem comportadas e tem coeficientes interpretáveis. Só agora usamos o teste.

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, ConfusionMatrixDisplay
# TODO 3 (cap. 5): treine a logística no treino inteiro e preveja o teste
final_c = None
pred_te = None
print(f'TESTE · F1 = {f1_score(yc_te, pred_te):.3f} | recall = {recall_score(yc_te, pred_te):.3f} | '
      f'precisão = {precision_score(yc_te, pred_te):.3f}')
ConfusionMatrixDisplay.from_predictions(yc_te, pred_te, display_labels=['benigno', 'maligno'], cmap='Blues')
plt.title('Logística · teste (usado uma vez)'); plt.show()

## Checklist da Unidade 3
- [ ] Comparei pelo menos 2 famílias de modelos com o mesmo `cv` e a mesma métrica.
- [ ] Sei dizer qual é o botão contra o sobreajuste de cada modelo que usei.
- [ ] Hiperparâmetros escolhidos por `GridSearchCV` só no treino.
- [ ] Li média **e** desvio entre dobras antes de declarar um vencedor.
- [ ] Em caso de empate, escolhi o modelo mais simples e explicável.
- [ ] Abri o teste uma única vez, depois da escolha.